# AIMO3 Judge-Based Selection (Feb 9) — 5+5+2 Architecture

## Strategy: 5+5+2 (Broad → Deep → 2 Judges)

Lighter variant of 7+7+2 (which timed out at 9h):
- Phase 1: 5 broad attempts (exit early if 3+ agree)
- Phase 2: 5 extra attempts (only if Phase 1 didn't converge)
- Phase 3: 2 judges evaluate ALL solutions, combined scoring

Key changes vs 772:
- 10 solver attempts (vs 14) — ~30% fewer inference calls
- Judge max tokens: 4096 (vs 16384) — 4× faster judges
- Trace chars for judge: 2000 (vs 6000) — smaller judge prompts
- notebook_limit: 25200 (7h, vs 8h) — safer margin
- Problem timeout: 720s (vs 960s) — tighter per-problem budget

Time estimate: ~1.5× feb3 (which ran ~5h) = ~7.5h, well within 9h limit.

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, gc, re, math, time, queue, threading, contextlib, json
from datetime import datetime

In [ ]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels',
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

In [ ]:
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd, polars as pl
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort,
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

## Configuration

5+5+2 architecture: Broad search → Deep search → 2 Judges with combined scoring.
JSON traces saved per-problem in local mode only.

In [ ]:
class CFG:
    # Solver prompt
    system_prompt = ('You are a world-class International Mathematical Olympiad (IMO) competitor. '
                    'The final answer must be a non-negative integer between 0 and 99999. '
                    'You must place the final integer answer inside \\boxed{}.')
    tool_prompt = ('Use this tool to execute Python code. The environment is a stateful Jupyter notebook. '
                  'You must use print() to output results.')
    preference_prompt = 'You have access to `math`, `numpy` and `sympy` to solve the problem.'

    # Judge prompt
    judge_system_prompt = ('You are reviewing solutions to a math competition problem. '
                          'The final answer must be a non-negative integer between 0 and 99999. '
                          'Place each answer inside \\boxed{}.')

    # Model
    served_model_name, model_path = 'gpt-oss', '/kaggle/input/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'

    # Timing (tighter for 552 — must finish in <9h)
    high_problem_timeout, base_problem_timeout = 720, 270
    notebook_limit, server_timeout = 25200, 180       # 7h self-limit (vs 8h in 772)
    session_timeout, jupyter_timeout, sandbox_timeout = 960, 6, 3

    # Generation
    stream_interval, context_tokens, buffer_tokens, search_tokens = 200, 65536, 512, 32
    top_logprobs, batch_size = 5, 256
    gpu_memory_utilization, temperature, min_p, seed = 0.96, 1.0, 0.02, 42

    # Phase 1: Broad search (5 attempts, exit if 3+ agree)
    phase1_attempts = 5
    phase1_workers = 5
    phase1_early_stop = 3

    # Phase 2: Deep search (5 attempts, only if Phase 1 doesn't converge)
    phase2_attempts = 5
    phase2_workers = 5

    # Phase 3: 2 judges with tight token budget
    num_judges = 2
    judge_temperature = 0.7
    judge_max_tokens = 4096       # 4× smaller than 772
    trace_max_chars = 2000        # 3× smaller than 772

    # Kernel pool & generation limits
    workers = 5
    turns = 128

    # Entropy-gated consensus (fallback if judges fail)
    entropy_threshold = 5.0
    min_consensus = 2

    # Tracing (local mode only)
    trace_dir = 'traces'
    is_competition = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

print(f"5+5+2 Judge Architecture | Competition mode: {CFG.is_competition}")
print(f"Solver: {CFG.phase1_attempts}+{CFG.phase2_attempts} attempts | Judges: {CFG.num_judges} x {CFG.judge_max_tokens} max tokens")
print(f"Time budget: {CFG.notebook_limit}s ({CFG.notebook_limit/3600:.1f}h) | Per-problem max: {CFG.high_problem_timeout}s")

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg=None):
        sc = SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH)
        if tool_cfg is not None:
            sc = sc.with_tools(tool_cfg)
        return sc

    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg=None):
        return [Message.from_role_and_content(Role.SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

In [ ]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb
                      if 'File "' not in f or 'ipython-input' in f)

    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output. Use print() to see results.')

    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()

    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def __del__(self):
        self.close()

In [ ]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox=None):
        self._local_jupyter_timeout, self._tool_prompt, self._jupyter_session = timeout, prompt, sandbox
        self._owns_session, self._execution_lock, self._init_lock = sandbox is None, threading.Lock(), threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)

    @property
    def instruction(self): return self._tool_prompt

    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])

    def _make_response(self, output, channel=None):
        msg = Message(author=Author(role=Role.TOOL, name='python'),
                     content=[TextContent(text=output)]).with_recipient('assistant')
        return msg.with_channel(channel) if channel else msg

    def process_sync_plus(self, message):
        self._ensure_session()
        final_script = self._ensure_last_print(message.content[0].text)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

## AIMO3 Solver with 2-Judge Authority (552 variant)

Three-phase architecture:
1. **Phase 1 (Broad)**: 5 parallel attempts, exit early if 3+ agree
2. **Phase 2 (Deep)**: 5 more attempts if Phase 1 didn't converge
3. **Phase 3 (Judges)**: 2 judges evaluate all solutions, each picks top 2

Key timing optimization: judges see only 2000 chars of trace (vs 6000) and
generate max 4096 tokens (vs 16384). This cuts judge time by ~4×.

In [ ]:
class AIMO3Solver:
    def __init__(self, cfg, port=8000):
        self.cfg, self.port = cfg, port
        self.base_url, self.api_key = f'http://0.0.0.0:{port}/v1', 'sk-local'
        self.template, self.encoding = AIMO3Template(), load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server()
        self._initialize_kernels()
        self.notebook_start_time, self.problems_remaining = time.time(), 50
        self.problem_counter = 0
        self.timing_log = []  # Per-problem timing for post-analysis
        if not self.cfg.is_competition:
            os.makedirs(self.cfg.trace_dir, exist_ok=True)

    def _preload_model_weights(self):
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start, files, total = time.time(), [], 0
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp):
                    files.append(fp)
                    total += os.path.getsize(fp)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Processed {len(files)} files ({total/1e9:.2f} GB) in {time.time()-start:.2f} seconds.\n')

    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--seed', str(self.cfg.seed),
               '--model', self.cfg.model_path, '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), '--host', '0.0.0.0',
               '--port', str(self.port), '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens), '--stream-interval', str(self.cfg.stream_interval),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if (rc := self.server_process.poll()) is not None:
                self.log_file.flush()
                raise RuntimeError(f'Server died with code {rc}. Full logs:\n{open("vllm_server.log").read()}\n')
            try:
                self.client.models.list()
                print(f'Server is ready (took {time.time()-start:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self):
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels (sequential with retry)...')
        start = time.time()
        self.sandbox_pool = queue.Queue()
        for i in range(self.cfg.workers):
            for attempt in range(3):
                try:
                    sandbox = AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
                    self.sandbox_pool.put(sandbox)
                    print(f'  Kernel {i+1}/{self.cfg.workers} ready')
                    break
                except Exception as e:
                    if attempt < 2:
                        print(f'  Kernel {i+1} attempt {attempt+1} failed, retrying...')
                        time.sleep(1)
                    else:
                        print(f'  Kernel {i+1} failed after 3 attempts: {e}')
        print(f'Kernels initialized in {time.time()-start:.2f} seconds ({self.sandbox_pool.qsize()} ready).\n')

    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except ValueError: pass
        return None

    def _scan_all_answers(self, text):
        answers = []
        for m in re.finditer(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text):
            try:
                val = int(m.group(1).replace(',', ''))
                if 0 <= val <= 99999: answers.append(val)
            except ValueError: pass
        return answers

    def _compute_mean_entropy(self, logprobs):
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for top_lp in logprobs:
            if isinstance(top_lp, dict) and top_lp:
                ent = sum(-math.exp(lp)*math.log2(math.exp(lp)) for lp in top_lp.values() if math.exp(lp) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')

    def _process_attempt(self, problem, sys_prompt, idx, stop_evt, deadline, use_tools=True):
        mode = 'tir' if use_tools else 'text_only'
        if stop_evt.is_set() or time.time() > deadline:
            return {'Attempt': idx+1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0,
                   'Response Length': 0, 'Entropy': float('inf'), 'turns': [], 'Duration': 0.0,
                   'model': self.cfg.served_model_name, 'mode': mode,
                   'token_budget': self.cfg.context_tokens, 'tokens_remaining': self.cfg.context_tokens}

        local_tool, sandbox = None, None
        py_calls, py_errs, total_toks, ans, logprobs = 0, 0, 0, None, []
        turns = []
        seed = int(math.pow(self.cfg.seed + idx, 2))
        attempt_start = time.time()
        try:
            if use_tools:
                sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
                local_tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, sandbox)
                tool_cfg = local_tool.tool_config
            else:
                tool_cfg = None

            conv = Conversation.from_messages(self.template.apply_chat_template(
                sys_prompt, problem, tool_cfg))
            for _ in range(self.cfg.turns):
                if stop_evt.is_set() or time.time() > deadline: break
                prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
                if (max_toks := self.cfg.context_tokens - len(prompt_ids)) < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature, logprobs=self.cfg.top_logprobs, max_tokens=max_toks,
                    prompt=prompt_ids, seed=seed, stream=True, extra_body={
                        'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True})
                try:
                    tok_buf, txt_chunks = [], []
                    for chunk in stream:
                        if stop_evt.is_set() or time.time() > deadline: break
                        if new_toks := chunk.choices[0].token_ids:
                            tok_buf.extend(new_toks)
                            total_toks += len(new_toks)
                            txt_chunks.append(chunk.choices[0].text)
                            if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                                logprobs.extend(clp.top_logprobs)
                        if '}' in chunk.choices[0].text and (ans := self._scan_for_answer(
                            ''.join(txt_chunks[-self.cfg.search_tokens:]))):
                            break
                finally:
                    stream.close()
                turn_text = ''.join(txt_chunks)
                if ans or not tok_buf:
                    if turn_text:
                        turns.append({'type': 'text', 'content': turn_text})
                    break
                new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                last = new_msgs[-1]
                if last.channel == 'final':
                    turns.append({'type': 'text', 'content': last.content[0].text})
                    ans = self._scan_for_answer(last.content[0].text)
                    break
                if use_tools and last.recipient == 'python':
                    turns.append({'type': 'text', 'content': turn_text})
                    py_calls += 1
                    code_text = last.content[0].text
                    resp = local_tool.process_sync_plus(last)
                    tool_output = resp[0].content[0].text
                    success = not any(x in tool_output for x in ['[ERROR]', 'Traceback', 'Error:'])
                    if not success: py_errs += 1
                    turns.append({'type': 'code', 'content': code_text[:2000]})
                    turns.append({'type': 'code_output', 'content': tool_output[:2000], 'success': success})
                    conv.messages.extend(resp)
                else:
                    turns.append({'type': 'text', 'content': turn_text})
        except Exception: py_errs += 1
        finally:
            if sandbox:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        tokens_remaining = self.cfg.context_tokens - total_toks
        return {'Attempt': idx+1, 'Response Length': total_toks, 'Python Calls': py_calls,
               'Python Errors': py_errs, 'Entropy': self._compute_mean_entropy(logprobs), 'Answer': ans,
               'turns': turns, 'Duration': round(time.time() - attempt_start, 2),
               'model': self.cfg.served_model_name, 'mode': mode,
               'token_budget': self.cfg.context_tokens, 'tokens_remaining': tokens_remaining}

    # ── Turns to text (for judge prompt) ──────────────────────────────────

    def _turns_to_text(self, turns, max_chars=None):
        if max_chars is None:
            max_chars = self.cfg.trace_max_chars
        parts = []
        for t in turns:
            if t['type'] == 'text':
                parts.append(t['content'])
            elif t['type'] == 'code':
                parts.append(f'\n```python\n{t["content"]}\n```\n')
            elif t['type'] == 'code_output':
                status = 'OK' if t.get('success', True) else 'ERROR'
                parts.append(f'[Output ({status})]: {t["content"][:500]}\n')
        full = ''.join(parts)
        if len(full) > max_chars:
            full = '...' + full[-max_chars:]
        return full

    # ── Phase execution ──────────────────────────────────────────────────

    def _run_phase(self, user_input, n_attempts, n_workers, early_stop, stop_evt, deadline, idx_offset=0, text_only_count=0):
        """Run N attempts in parallel. Last text_only_count attempts use text-only mode."""
        results, valid = [], []
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futures = []
            for i in range(n_attempts):
                use_tools = i < (n_attempts - text_only_count)
                futures.append(ex.submit(self._process_attempt, user_input, self.cfg.system_prompt,
                                         idx_offset + i, stop_evt, deadline, use_tools))
            for future in as_completed(futures):
                try:
                    if (r := future.result())['Answer'] is not None:
                        valid.append(r['Answer'])
                    results.append(r)
                    if early_stop and (cnts := Counter(valid).most_common(1)) and cnts[0][1] >= early_stop:
                        stop_evt.set()
                        for f in futures: f.cancel()
                        return results, True
                except Exception as exc:
                    print(f'Future failed: {exc}')
        return results, False

    # ── Judge ─────────────────────────────────────────────────────────────

    def _build_judge_prompt(self, problem, results):
        valid = [r for r in results if r['Answer'] is not None]
        if not valid: return None

        groups = defaultdict(list)
        for r in valid:
            groups[r['Answer']].append(r)

        total = len(valid)
        parts = [
            f"{total} solvers attempted the following math competition problem:\n",
            "PROBLEM:", problem,
            f"\nTheir answers, grouped by frequency:\n"
        ]
        for answer, members in sorted(groups.items(), key=lambda x: -len(x[1])):
            count = len(members)
            best = min(members, key=lambda r: r['Entropy'])
            parts.append(f"<ANSWER {answer} ({count}/{total} solvers)>")
            trace_text = self._turns_to_text(best.get('turns', []))
            if trace_text:
                parts.append(trace_text)
            parts.append(f"</ANSWER>\n")

        parts.append(
            "Study the solutions above. Assess the reasoning quality and determine which "
            "answers are most likely correct. Output two answers you believe are correct "
            "for this problem, most confident first. Place each inside \\boxed{}.\n\n"
            "You may choose from the candidate answers or derive your own if you believe "
            "the solutions contain errors. Explain your meta-conclusion briefly."
        )
        return '\n'.join(parts)

    def _run_judge(self, problem, results, deadline, judge_idx=0):
        """Run judge. Returns (list of up to 2 picked answers, judge_trace)."""
        judge_trace = {'judge_index': judge_idx}
        if time.time() > deadline:
            return [], judge_trace

        judge_prompt = self._build_judge_prompt(problem, results)
        if judge_prompt is None:
            return [], judge_trace

        seed = self.cfg.seed + 1000 + judge_idx * 100
        conv = Conversation.from_messages(self.template.apply_chat_template(
            self.cfg.judge_system_prompt, judge_prompt))
        prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
        max_toks = min(self.cfg.judge_max_tokens, self.cfg.context_tokens - len(prompt_ids))

        judge_trace['prompt_tokens'] = len(prompt_ids)
        if not self.cfg.is_competition:
            judge_trace['prompt'] = judge_prompt

        if max_toks < 256:
            print(f'Judge {judge_idx+1}: prompt too long ({len(prompt_ids)} tokens), skipping')
            judge_trace['error'] = 'prompt_too_long'
            return [], judge_trace

        start_ts = time.time()
        try:
            stream = self.client.completions.create(
                model=self.cfg.served_model_name,
                temperature=self.cfg.judge_temperature,
                max_tokens=max_toks,
                prompt=prompt_ids,
                seed=seed,
                stream=True,
                extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids})
            text_chunks = []
            try:
                for chunk in stream:
                    if time.time() > deadline: break
                    text_chunks.append(chunk.choices[0].text)
            finally:
                stream.close()
            text = ''.join(text_chunks)
            duration = time.time() - start_ts

            answers = self._scan_all_answers(text)
            seen, unique = set(), []
            for a in answers:
                if a not in seen:
                    seen.add(a)
                    unique.append(a)

            judge_trace['duration_seconds'] = round(duration, 2)
            judge_trace['response_chars'] = len(text)
            judge_trace['response'] = text
            judge_trace['picked_answers'] = unique[:2]

            print(f'Judge {judge_idx+1}: picked {unique[:2]} ({len(text)} chars, {duration:.1f}s)')
            return unique[:2], judge_trace
        except Exception as e:
            print(f'Judge {judge_idx+1} error: {e}')
            judge_trace['error'] = str(e)
            return [], judge_trace

    # ── Combined scoring: multi-judge weighted picks ──────────────────────

    def _score_with_judges(self, results, all_judge_picks):
        """Score using ONLY judge picks (1st=2pts, 2nd=1pt per judge)."""
        scores = defaultdict(int)
        for judge_picks in all_judge_picks:
            if len(judge_picks) >= 1:
                scores[judge_picks[0]] += 2
            if len(judge_picks) >= 2:
                scores[judge_picks[1]] += 1
        return dict(scores)

    # ── Entropy-gated consensus (fallback) ────────────────────────────────

    def _select_answer_gated(self, results):
        valid_results = [r for r in results if r['Answer'] is not None]
        if not valid_results: return 0
        confident = [r for r in valid_results if r['Entropy'] < self.cfg.entropy_threshold]
        if confident:
            votes = Counter(r['Answer'] for r in confident)
            candidates = {a: c for a, c in votes.items() if c >= self.cfg.min_consensus}
            if candidates:
                scores = defaultdict(float)
                for r in confident:
                    if r['Answer'] in candidates:
                        scores[r['Answer']] += 1.0 / max(r['Entropy'], 0.1)
                winner = max(scores, key=scores.get)
                print(f'Fallback (entropy-gated): {winner}')
                return winner
        winner = Counter(r['Answer'] for r in valid_results).most_common(1)[0][0]
        print(f'Fallback (majority): {winner}')
        return winner

    # ── Trace helpers ─────────────────────────────────────────────────────

    def _attempt_to_trace(self, r, include_full=False):
        d = {k: v for k, v in r.items() if k != 'turns'}
        if include_full:
            d['turns'] = r.get('turns', [])
        return d

    def _save_trace(self, trace_data):
        if self.cfg.is_competition:
            return
        pid = trace_data.get('problem_index', self.problem_counter)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        path = os.path.join(self.cfg.trace_dir, f'{ts}_problem_{pid:03d}.json')
        try:
            with open(path, 'w') as f:
                json.dump(trace_data, f, indent=2, default=str)
            print(f'Trace saved: {path}')
        except Exception as e:
            print(f'Trace save failed: {e}')

    # ── Timing summary ───────────────────────────────────────────────────

    def _print_timing_summary(self):
        """Print cumulative timing summary after each problem."""
        if not self.timing_log: return
        elapsed = time.time() - self.notebook_start_time
        total_problem_time = sum(t['seconds'] for t in self.timing_log)
        avg_time = total_problem_time / len(self.timing_log)
        remaining = 50 - len(self.timing_log)
        projected = elapsed + remaining * avg_time

        early_pct = sum(1 for t in self.timing_log if t['method'] == 'early_stop') / len(self.timing_log) * 100
        judge_pct = sum(1 for t in self.timing_log if 'judge' in t['method']) / len(self.timing_log) * 100

        print(f'\n>>> TIMING: {len(self.timing_log)}/50 done | '
              f'Elapsed: {elapsed/3600:.2f}h | '
              f'Avg: {avg_time:.0f}s/problem | '
              f'Projected total: {projected/3600:.2f}h | '
              f'Early-stop: {early_pct:.0f}% | Judge: {judge_pct:.0f}%')
        if projected > 30600:  # 8.5h warning
            print(f'>>> WARNING: Projected {projected/3600:.1f}h may exceed 9h limit!')

    # ── Main solve ────────────────────────────────────────────────────────

    def solve_problem(self, problem):
        self.problem_counter += 1
        problem_start = time.time()
        print(f'\n{"="*60}')
        print(f'Problem {self.problem_counter}: {problem[:100]}...\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        time_left = self.cfg.notebook_limit - (time.time() - self.notebook_start_time)
        budget = max(self.cfg.base_problem_timeout,
                    min(time_left - max(0, self.problems_remaining-1)*self.cfg.base_problem_timeout,
                        self.cfg.high_problem_timeout))
        deadline = time.time() + budget
        n_judges = self.cfg.num_judges
        print(f'Budget: {budget:.0f}s | Phases: {self.cfg.phase1_attempts}+{self.cfg.phase2_attempts}+{n_judges}\n')

        local = not self.cfg.is_competition

        trace = {
            'problem_index': self.problem_counter,
            'problem_text': problem,
            'timestamp': datetime.now().isoformat(),
            'budget_seconds': round(budget, 1),
            'config': {
                'model': self.cfg.served_model_name,
                'temperature': self.cfg.temperature,
                'phase1_attempts': self.cfg.phase1_attempts,
                'phase2_attempts': self.cfg.phase2_attempts,
                'phase1_early_stop': self.cfg.phase1_early_stop,
                'num_judges': n_judges,
                'entropy_threshold': self.cfg.entropy_threshold,
                'judge_temperature': self.cfg.judge_temperature,
            },
            'phases': {},
        }

        stop_evt = threading.Event()

        # ── Phase 1: 3 TIR + 2 text-only ──
        p1_text_only = min(2, self.cfg.phase1_attempts)
        print(f'--- Phase 1: {self.cfg.phase1_attempts} attempts ({self.cfg.phase1_attempts - p1_text_only} TIR + {p1_text_only} text-only) ---')
        p1_start = time.time()
        results, early_stopped = self._run_phase(
            user_input, self.cfg.phase1_attempts, self.cfg.phase1_workers,
            self.cfg.phase1_early_stop, stop_evt, deadline,
            text_only_count=p1_text_only)

        if results:
            df = pd.DataFrame([self._attempt_to_trace(r) for r in results])
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df)

        trace['phases']['phase1'] = {
            'duration_seconds': round(time.time() - p1_start, 2),
            'early_stopped': early_stopped,
            'attempts': [self._attempt_to_trace(r, include_full=local) for r in results],
        }

        self.problems_remaining = max(0, self.problems_remaining - 1)
        valid = [r['Answer'] for r in results if r['Answer'] is not None]

        if early_stopped:
            winner = Counter(valid).most_common(1)[0][0]
            elapsed = round(time.time() - problem_start, 2)
            print(f'\nPhase 1 early stop: {winner} (consensus >= {self.cfg.phase1_early_stop}) [{elapsed:.0f}s]\n')
            trace['final_answer'] = winner
            trace['selection_method'] = 'early_stop'
            trace['total_seconds'] = elapsed
            self._save_trace(trace)
            self.timing_log.append({'problem': self.problem_counter, 'seconds': elapsed, 'method': 'early_stop', 'answer': winner})
            self._print_timing_summary()
            return winner

        if not valid:
            elapsed = round(time.time() - problem_start, 2)
            print(f'No valid answers from Phase 1, returning 0 [{elapsed:.0f}s]')
            trace['final_answer'] = 0
            trace['selection_method'] = 'no_valid'
            trace['total_seconds'] = elapsed
            self._save_trace(trace)
            self.timing_log.append({'problem': self.problem_counter, 'seconds': elapsed, 'method': 'no_valid', 'answer': 0})
            self._print_timing_summary()
            return 0

        # ── Phase 2: 3 TIR + 2 text-only ──
        if time.time() < deadline:
            p2_text_only = min(2, self.cfg.phase2_attempts)
            print(f'\n--- Phase 2: {self.cfg.phase2_attempts} attempts ({self.cfg.phase2_attempts - p2_text_only} TIR + {p2_text_only} text-only) ---')
            p2_start = time.time()
            stop_evt.clear()
            more_results, _ = self._run_phase(
                user_input, self.cfg.phase2_attempts, self.cfg.phase2_workers,
                None, stop_evt, deadline, idx_offset=self.cfg.phase1_attempts,
                text_only_count=p2_text_only)
            results.extend(more_results)

            if more_results:
                df2 = pd.DataFrame([self._attempt_to_trace(r) for r in more_results])
                df2['Entropy'] = df2['Entropy'].round(3)
                df2['Answer'] = df2['Answer'].astype('Int64')
                display(df2)

            trace['phases']['phase2'] = {
                'duration_seconds': round(time.time() - p2_start, 2),
                'attempts': [self._attempt_to_trace(r, include_full=local) for r in more_results],
            }

        all_valid = [r for r in results if r['Answer'] is not None]
        all_votes = Counter(r['Answer'] for r in all_valid)
        print(f'\nAll candidates ({len(all_valid)} valid): {dict(all_votes.most_common())}')
        trace['candidates'] = {str(a): c for a, c in all_votes.most_common()}

        # ── Phase 3: Multiple Judges + combined scoring ──
        all_judge_picks = []
        judge_traces = []
        if time.time() < deadline:
            print(f'\n--- Phase 3: {n_judges} Judge{"s" if n_judges > 1 else ""} ---')
            for j_idx in range(n_judges):
                if time.time() > deadline:
                    break
                judge_picks, judge_trace = self._run_judge(user_input, results, deadline, judge_idx=j_idx)
                judge_traces.append(judge_trace)
                if judge_picks:
                    all_judge_picks.append(judge_picks)
            trace['phases']['judges'] = judge_traces

            if all_judge_picks:
                combined = self._score_with_judges(results, all_judge_picks)
                winner = max(combined, key=combined.get)
                elapsed = round(time.time() - problem_start, 2)
                print(f'Combined scores: {dict(sorted(combined.items(), key=lambda x: -x[1]))}')
                print(f'Winner: {winner} [{elapsed:.0f}s]\n')
                trace['combined_scores'] = {str(k): v for k, v in combined.items()}
                trace['final_answer'] = winner
                trace['selection_method'] = 'judge_combined'
                trace['total_seconds'] = elapsed
                self._save_trace(trace)
                self.timing_log.append({'problem': self.problem_counter, 'seconds': elapsed, 'method': 'judge_combined', 'answer': winner})
                self._print_timing_summary()
                return winner

        # Fallback
        print('\n--- Fallback: entropy-gated consensus ---')
        fallback = self._select_answer_gated(results)
        elapsed = round(time.time() - problem_start, 2)
        trace['final_answer'] = fallback
        trace['selection_method'] = 'fallback_entropy'
        trace['total_seconds'] = elapsed
        self._save_trace(trace)
        self.timing_log.append({'problem': self.problem_counter, 'seconds': elapsed, 'method': 'fallback_entropy', 'answer': fallback})
        self._print_timing_summary()
        return fallback

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'): self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                with contextlib.suppress(Exception): self.sandbox_pool.get_nowait().close()

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    gc.disable()
    final_answer = solver.solve_problem(question.item(0))
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': id_.item(0), 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',))